# Project: SMS Spam Filter 📵
Build a **Naive Bayes** spam filter from scratch (counting words, nothing else), check it against scikit-learn, and then decide where to put the threshold when blocking a real message is worse than letting spam through.

Topic: [[Naive Bayes]] · guide note: [[Project - SMS Spam Filter]] · data: the [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) (5,574 English SMS, 13 % spam)

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import io, pathlib, re, urllib.request, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

f = pathlib.Path("data/SMSSpamCollection")
if not f.exists():
    f.parent.mkdir(exist_ok=True)
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip").read())).extract("SMSSpamCollection", "data")
sms = pd.read_csv(f, sep="\t", header=None, names=["label", "text"], quoting=3)
tokenize = lambda s: re.findall(r"[a-z0-9£$]+", s.lower())
docs = [tokenize(t) for t in sms.text]
d_tr, d_te, y_tr, y_te = train_test_split(docs, sms.label.tolist(), test_size=0.25, stratify=sms.label, random_state=0)
print(sms.label.value_counts().to_dict(), "| example spam:", sms.text[sms.label == "spam"].iloc[0][:90])

## 1. Training = counting
Multinomial Naive Bayes assumes words are independent given the class:
$$\log P(c\mid d) = \log P(c) + \sum_{t\in d}\log P(t\mid c) + \text{const},\qquad P(t\mid c)=\frac{\text{count}(t,c)+\alpha}{\sum_{t'}\text{count}(t',c)+\alpha|V|}$$
The $+\alpha$ (Laplace smoothing) stops one unseen word from making a probability zero. Return a dict with `classes`, `idx` (word → column), `log_prior[c]` and `log_lik[c]` (an array over the vocabulary).

In [ ]:
def fit_nb(docs, labels, alpha=1.0):
    # TODO 1: vocabulary from training docs; per class: smoothed word counts → log probabilities; log class priors
    raise NotImplementedError  # TODO 1

## 2. Prediction = adding logs
Return `{class: log P(c) + Σ log P(t|c)}` for one document, **skipping words not in the vocabulary**.

In [ ]:
def score(model, doc):
    # TODO 2: unnormalised log posterior per class
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_nb():
    m = fit_nb([["win", "cash"], ["win", "now"], ["see", "you"]], ["spam", "spam", "ham"], alpha=1.0)
    s = score(m, ["win", "unknownword"])
    # spam: 4 tokens, vocab 5 → P(win|spam) = (2+1)/(4+5); ham: P(win|ham) = (0+1)/(2+5)
    return (abs(s["spam"] - (np.log(2 / 3) + np.log(3 / 9))) < 1e-9 and abs(s["ham"] - (np.log(1 / 3) + np.log(1 / 7))) < 1e-9)
check("naive bayes", _t_nb, "Toy: P(win|spam) = 3/9, P(win|ham) = 1/7, priors 2/3 and 1/3; unknown words are skipped.")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score, recall_score, f1_score
if ready("naive bayes"):
    model = fit_nb(d_tr, y_tr)
    margin = np.array([score(model, d)["spam"] - score(model, d)["ham"] for d in d_te])       # log-odds of spam
    pred = np.where(margin > 0, "spam", "ham")
    vec = CountVectorizer(tokenizer=lambda d: d, preprocessor=lambda d: d, token_pattern=None)
    sk = MultinomialNB(alpha=1.0).fit(vec.fit_transform(d_tr), y_tr).predict(vec.transform(d_te))
    p, r, f1 = (fn(y_te, pred, pos_label="spam") for fn in (precision_score, recall_score, f1_score))
    print(f"accuracy {np.mean(pred == np.array(y_te)):.4f} (sklearn {np.mean(sk == np.array(y_te)):.4f}) | spam precision {p:.3f} recall {r:.3f} F1 {f1:.3f}")
    check("matches scikit-learn", lambda: np.mean(pred == sk) > 0.999 and f1 > 0.9, "Your predictions should match MultinomialNB(alpha=1) and reach spam-F1 > 0.9.")

## 3. What does it look for?
The most spam-indicative words have the largest $\log P(t\mid\text{spam}) - \log P(t\mid\text{ham})$. Return the top `n` words.

In [ ]:
def spammiest_words(model, n=15):
    # TODO 3: words sorted by log-likelihood ratio spam vs ham, descending
    raise NotImplementedError  # TODO 3

In [ ]:
check("spammy words", lambda: len(set(spammiest_words(fit_nb(d_tr, y_tr), 30)) & {"claim", "prize", "guaranteed", "awarded", "ringtone", "150p", "£1000", "entry", "tone", "txt", "won", "urgent"}) >= 3,
      "The top-30 should contain classic spam words like claim, prize, guaranteed, awarded.")
if ready("naive bayes", "spammy words"):
    print("spammiest:", spammiest_words(model, 20))

## 4. Choosing the threshold
Blocking a friend's message is far worse than one spam getting through. Slide the threshold on the log-odds and look at the trade-off. What's the best recall you can get with **zero** blocked real messages on the test set?

In [ ]:
if ready("naive bayes"):
    yt = np.array(y_te) == "spam"; ts = np.linspace(-10, 30, 200)
    fpr = [np.mean(margin[~yt] > t) for t in ts]; rec = [np.mean(margin[yt] > t) for t in ts]
    plt.plot(ts, rec, label="spam caught (recall)"); plt.plot(ts, fpr, label="real messages blocked (FPR)"); plt.axvline(0, ls="--", c="gray")
    plt.xlabel("threshold on log-odds"); plt.legend(); plt.show()
    t0 = margin[~yt].max(); print(f"threshold {t0:.1f} blocks 0 real messages and still catches {np.mean(margin[yt] > t0):.0%} of spam")

<details><summary>▶ Solution</summary>

```python
def fit_nb(docs, labels, alpha=1.0):
    classes = sorted(set(labels))
    vocab = sorted({t for d in docs for t in d})
    idx = {w: i for i, w in enumerate(vocab)}
    log_prior, log_lik = {}, {}
    for c in classes:
        counts = np.full(len(vocab), alpha)
        for d, y in zip(docs, labels):
            if y == c:
                for t in d:
                    counts[idx[t]] += 1
        log_lik[c] = np.log(counts / counts.sum())
        log_prior[c] = np.log(sum(y == c for y in labels) / len(labels))
    return {"classes": classes, "idx": idx, "log_prior": log_prior, "log_lik": log_lik}
```

```python
def score(model, doc):
    return {c: model["log_prior"][c] + sum(model["log_lik"][c][model["idx"][t]] for t in doc if t in model["idx"])
            for c in model["classes"]}
```

```python
def spammiest_words(model, n=15):
    ratio = model["log_lik"]["spam"] - model["log_lik"]["ham"]
    inv = {i: w for w, i in model["idx"].items()}
    return [inv[i] for i in np.argsort(-ratio)[:n]]
```
</details>

## Stretch goals
- Compare with logistic regression on the same counts ([[Logistic Regression]]): who wins with 100 training messages? With all of them? (Generative vs. discriminative, Ng & Jordan.)
- Add character n-grams to catch obfuscations like "fr33".
- Calibrate: are Naive Bayes probabilities trustworthy? Plot a reliability diagram ([[Model Evaluation and Metrics]]).